# Experimento final — Reconocimiento de Patrones
## Tomek Links + D-min en un espacio multiclase altamente asimétrico

Este notebook está diseñado para ejecutar el experimento que posteriormente se incorporará a la tesina.

### Decisiones metodológicas fijadas

- **N = 2000** observaciones.
- **10 características**.
- **3 clases**, para cumplir el planteamiento multiclase:
  - \(\omega_0\): clase de referencia/mayoritaria.
  - \(\omega_1\): patrón intermedio/no objetivo.
  - \(\omega_2\): clase patológica rara.
- Proporciones aproximadas:
  \[
  P(\omega_0)=0.982,\qquad P(\omega_1)=0.010,\qquad P(\omega_2)=0.008.
  \]
  Por tanto:
  \[
  P(\omega_2)<1\%.
  \]
- La evaluación clínica se realiza **one-vs-rest**:
  \[
  \omega_2\quad\text{vs}\quad\{\omega_0,\omega_1\}.
  \]
- Normalización principal: **mediana/MAD**, estimada únicamente en entrenamiento.
- Tomek Links: aplicado únicamente al **training fold** y eliminando sólo miembros de la clase mayoritaria.
- Control experimental: **random undersampling emparejado en cantidad** con Tomek.
- Benchmark secundario: SMOTE+Tomek.
- D-min multiclase con 6 medidas de distancia.
- Validación: **Repeated Stratified K-Fold**; \(K\) se determina a partir del número real de casos raros y \(R=10\).
- Las métricas predictivas se calculan sobre **predicciones out-of-fold acumuladas dentro de cada repetición**, no promediando métricas no lineales de folds.
- Inferencia confirmatoria: comparación pareada Baseline vs Tomek sobre:
  - \(G_{\mathrm{rara}}\)
  - sensibilidad
  - exactitud balanceada
  - ROC-AUC
- Wilcoxon unilateral + Holm + bootstrap descriptivo de las diferencias por repetición.
- kDN se añade como **análisis local complementario**, sin modificar las hipótesis principales.

### Fuentes que fundamentan el diseño

- Tomek (1976): enlaces de vecinos recíprocos de clases opuestas.
- Batista, Prati & Monard (2004): solapamiento, Tomek, ENN y remuestreo.
- Rousseeuw & Croux (1993): MAD robusto y constante 1.4826.
- Ledoit & Wolf (2004): covarianza shrinkage bien condicionada.
- Duda, Hart & Stork; Theodoridis et al.: derivación Bayes \(\rightarrow\) D-min.
- Brodersen et al. (2010): exactitud balanceada.
- Saito & Rehmsmeier (2015): Precision–Recall en fuerte desbalance.
- Varma & Simon (2006): evitar fuga de información en validación.


In [ ]:
# ============================================================
# 1. Instalación
# ============================================================
!pip -q install imbalanced-learn statsmodels


In [ ]:
# ============================================================
# 2. Importaciones y semilla
# ============================================================
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    accuracy_score
)
from sklearn.covariance import LedoitWolf
from sklearn.neighbors import NearestNeighbors

from imblearn.under_sampling import TomekLinks
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE

from scipy.spatial.distance import cdist
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

SEED = 42
np.random.seed(SEED)

RESULTS_DIR = Path("/content/resultados_tesina")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Directorio de resultados:", RESULTS_DIR)


# 3. Generación del conjunto experimental

Se emplea `make_classification` porque permite controlar dimensionalidad, distribución de clases, número de clusters y solapamiento.

La clase rara se fija en 0.8 %:

\[
n_{\mathrm{rara}}\approx N(0.008)=2000(0.008)=16.
\]

No se utiliza `flip_y` como fuente principal de ruido porque el planteamiento se refiere a **ruido geométrico/artefactos de medición**, no necesariamente a errores de etiqueta.

Se añaden perturbaciones de cola pesada a una fracción reducida de observaciones:

\[
x'_{ij}=x_{ij}+\varepsilon_{ij},
\qquad
\varepsilon_{ij}\sim t_{\nu=2}.
\]

Esto crea valores extremos de forma explícita y permite justificar el escalado robusto.


In [ ]:
# ============================================================
# 3. Configuración
# ============================================================
N_SAMPLES = 2000
N_FEATURES = 10

CLASS_WEIGHTS = [0.982, 0.010, 0.008]
RARE_CLASS = 2

N_INFORMATIVE = 6
N_REDUNDANT = 2
CLASS_SEP = 0.65

# Artefactos geométricos
OUTLIER_ROW_RATE = 0.02
OUTLIER_FEATURE_RATE = 0.20
OUTLIER_STRENGTH = 6.0

# Validación
MIN_RARE_PER_TEST_FOLD = 5
MAX_FOLDS = 5
N_REPEATS = 10

# kDN
K_KDN = 5

# Seis métricas estrictamente usadas para D-min/geometría
METRICS = [
    "euclidean",
    "cityblock",       # Manhattan
    "minkowski_p3",
    "chebyshev",
    "canberra",
    "mahalanobis"
]

PRIMARY_DISTANCE = "euclidean"

METHODS = [
    "baseline",
    "tomek",
    "random_matched",
    "smote_tomek"
]

CONFIG = {
    "seed": SEED,
    "N": N_SAMPLES,
    "d": N_FEATURES,
    "class_weights": CLASS_WEIGHTS,
    "rare_class": RARE_CLASS,
    "class_sep": CLASS_SEP,
    "outlier_row_rate": OUTLIER_ROW_RATE,
    "outlier_feature_rate": OUTLIER_FEATURE_RATE,
    "outlier_strength": OUTLIER_STRENGTH,
    "repeats": N_REPEATS,
    "metrics": METRICS,
    "primary_distance": PRIMARY_DISTANCE,
    "methods": METHODS
}


In [ ]:
# ============================================================
# 4. Dataset sintético multiclase
# ============================================================
X, y = make_classification(
    n_samples=N_SAMPLES,
    n_features=N_FEATURES,
    n_informative=N_INFORMATIVE,
    n_redundant=N_REDUNDANT,
    n_repeated=0,
    n_classes=3,
    n_clusters_per_class=1,
    weights=CLASS_WEIGHTS,
    class_sep=CLASS_SEP,
    flip_y=0.0,
    shuffle=False,  # permite usar las primeras características para visualización sin PCA
    random_state=SEED
)

# Inyección explícita de artefactos geométricos
rng = np.random.default_rng(SEED)

n_outlier_rows = max(1, int(round(N_SAMPLES * OUTLIER_ROW_RATE)))
outlier_rows = rng.choice(N_SAMPLES, size=n_outlier_rows, replace=False)

n_outlier_features = max(1, int(round(N_FEATURES * OUTLIER_FEATURE_RATE)))
feature_sd = X.std(axis=0, ddof=1)

for row in outlier_rows:
    feat_idx = rng.choice(
        N_FEATURES,
        size=n_outlier_features,
        replace=False
    )
    perturb = (
        rng.standard_t(df=2, size=n_outlier_features)
        * OUTLIER_STRENGTH
        * feature_sd[feat_idx]
    )
    X[row, feat_idx] += perturb

counts = pd.Series(y).value_counts().sort_index()
props = counts / len(y)

dataset_summary = pd.DataFrame({
    "clase": counts.index,
    "n": counts.values,
    "proporcion": props.values
})

display(dataset_summary)

n_rare = int(np.sum(y == RARE_CLASS))
print("Casos raros:", n_rare)
print("Prevalencia rara:", n_rare / len(y))
print("Artefactos geométricos inyectados:", len(outlier_rows))

dataset_summary.to_csv(RESULTS_DIR / "00_resumen_dataset.csv", index=False)


# 5. Normalización robusta Mediana/MAD

Para cada característica \(j\):

\[
\tilde{x}_j=\operatorname{mediana}(x_{1j},\ldots,x_{nj})
\]

\[
MAD_j=\operatorname{mediana}_i|x_{ij}-\tilde{x}_j|
\]

\[
s^{(R)}_j=1.4826\,MAD_j
\]

\[
z^{(R)}_{ij}=
\frac{x_{ij}-\tilde{x}_j}{s^{(R)}_j}.
\]

El factor:

\[
1.4826\approx\frac{1}{\Phi^{-1}(0.75)}
\]

hace que el MAD sea comparable con \(\sigma\) bajo una Normal de referencia.

**Regla de validez interna:** mediana y MAD se calculan sólo con el entrenamiento de cada fold. El conjunto de validación nunca participa en su estimación.


In [ ]:
# ============================================================
# 5. Escalador Mediana/MAD
# ============================================================
class MedianMADScaler:
    def __init__(self, consistency_constant=1.4826):
        self.c = consistency_constant

    def fit(self, X):
        X = np.asarray(X, dtype=float)

        self.center_ = np.median(X, axis=0)

        mad = np.median(
            np.abs(X - self.center_),
            axis=0
        )

        scale = self.c * mad

        # Fallback robusto si MAD = 0
        q75 = np.percentile(X, 75, axis=0)
        q25 = np.percentile(X, 25, axis=0)
        iqr_sigma = (q75 - q25) / 1.349

        scale = np.where(scale > 1e-12, scale, iqr_sigma)
        scale = np.where(scale > 1e-12, scale, 1.0)

        self.scale_ = scale
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return (X - self.center_) / self.scale_

    def fit_transform(self, X):
        return self.fit(X).transform(X)


# 6. Selección de folds

Con \(n_r\) casos raros y \(K\) folds:

\[
E[n_{r,\mathrm{test}}]\approx\frac{n_r}{K}.
\]

La sensibilidad es:

\[
Se=\frac{TP}{TP+FN}.
\]

Si un fold contiene \(m\) casos raros, cambiar una sola predicción puede modificar sensibilidad aproximadamente en:

\[
\frac{1}{m}.
\]

Se intenta mantener aproximadamente 5 casos raros por fold:

\[
K=
\max\left[
2,\;
\min\left(
5,\;
\left\lfloor
\frac{n_r}{5}
\right\rfloor
\right)
\right].
\]

Para \(n_r\approx16\), se obtiene \(K=3\).


In [ ]:
# ============================================================
# 6. K automático
# ============================================================
K = min(MAX_FOLDS, n_rare // MIN_RARE_PER_TEST_FOLD)
K = max(2, K)

CONFIG["folds"] = int(K)

print("K seleccionado:", K)
print("R:", N_REPEATS)
print("Folds totales:", K * N_REPEATS)
print("Casos raros esperados por test fold:", n_rare / K)
print("Resolución aproximada por un caso raro:", 1 / (n_rare / K))

with open(RESULTS_DIR / "00_configuracion_experimento.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2, ensure_ascii=False)


# 7. Geometría multiclase

Para cada clase \(k\):

\[
\mu_k=\frac{1}{n_k}\sum_{i:y_i=k}x_i
\]

\[
D_{\mathrm{intra},k}
=
\frac{1}{n_k}
\sum_{i:y_i=k}
d(x_i,\mu_k).
\]

Para evitar que la mayoría domine el indicador:

\[
D_{\mathrm{intra,bal}}
=
\frac{1}{C}
\sum_{k=1}^{C}D_{\mathrm{intra},k}.
\]

Separación global entre centroides:

\[
D_{\mathrm{inter,macro}}
=
\frac{2}{C(C-1)}
\sum_{a<b}
d(\mu_a,\mu_b).
\]

Separación específica de la clase rara \(r\):

\[
D_{\mathrm{inter,rara}}
=
\frac{1}{C-1}
\sum_{j\ne r}
d(\mu_r,\mu_j).
\]

Se reportan dos razones:

\[
G_{\mathrm{macro}}
=
\frac{D_{\mathrm{inter,macro}}}
{D_{\mathrm{intra,bal}}}
\]

y, como indicador clínico-geométrico principal:

\[
G_{\mathrm{rara}}
=
\frac{D_{\mathrm{inter,rara}}}
{D_{\mathrm{intra,rara}}}.
\]

Así se distinguen la geometría global y la separabilidad específica de la clase patológica rara.


In [ ]:
# ============================================================
# 7. Distancias y geometría
# ============================================================
def inverse_covariance_ledoit(X):
    lw = LedoitWolf().fit(X)
    return np.linalg.inv(lw.covariance_)

def distance_matrix(XA, XB, metric, VI=None):
    if metric == "euclidean":
        return cdist(XA, XB, metric="euclidean")
    if metric == "cityblock":
        return cdist(XA, XB, metric="cityblock")
    if metric == "minkowski_p3":
        return cdist(XA, XB, metric="minkowski", p=3)
    if metric == "chebyshev":
        return cdist(XA, XB, metric="chebyshev")
    if metric == "canberra":
        return cdist(XA, XB, metric="canberra")
    if metric == "mahalanobis":
        if VI is None:
            raise ValueError("Mahalanobis requiere VI.")
        return cdist(XA, XB, metric="mahalanobis", VI=VI)
    raise ValueError(metric)

def class_centroids(X, y):
    classes = np.unique(y)
    centroids = np.vstack([
        X[y == c].mean(axis=0)
        for c in classes
    ])
    return classes, centroids

def geometry_metrics(X, y, rare_class=RARE_CLASS, metric="euclidean", VI=None):
    classes, centroids = class_centroids(X, y)

    intra = {}
    for i, c in enumerate(classes):
        Xc = X[y == c]
        mu = centroids[i:i+1]
        distances = distance_matrix(Xc, mu, metric, VI).ravel()
        intra[int(c)] = float(np.mean(distances))

    D_intra_bal = float(np.mean(list(intra.values())))

    # Separaciones entre todos los centroides
    pairwise = []
    for i in range(len(classes)):
        for j in range(i+1, len(classes)):
            dij = distance_matrix(
                centroids[i:i+1],
                centroids[j:j+1],
                metric,
                VI
            )[0,0]
            pairwise.append(float(dij))

    D_inter_macro = float(np.mean(pairwise))

    # Separación específica de la clase rara
    rare_idx = np.where(classes == rare_class)[0][0]
    rare_dists = []

    for j in range(len(classes)):
        if j == rare_idx:
            continue
        rare_dists.append(
            float(
                distance_matrix(
                    centroids[rare_idx:rare_idx+1],
                    centroids[j:j+1],
                    metric,
                    VI
                )[0,0]
            )
        )

    D_inter_rare = float(np.mean(rare_dists))
    D_intra_rare = float(intra[int(rare_class)])

    G_macro = D_inter_macro / D_intra_bal if D_intra_bal > 0 else np.nan
    G_rare = D_inter_rare / D_intra_rare if D_intra_rare > 0 else np.nan

    result = {
        "D_intra_bal": D_intra_bal,
        "D_intra_rare": D_intra_rare,
        "D_inter_macro": D_inter_macro,
        "D_inter_rare": D_inter_rare,
        "G_macro": G_macro,
        "G_rare": G_rare
    }

    for c, val in intra.items():
        result[f"D_intra_class_{c}"] = val

    return result


# 8. kDN: ambigüedad local complementaria

Para cada observación:

\[
kDN(x_i)=
\frac{
|\{x_j\in N_k(x_i):y_j\ne y_i\}|
}{k}.
\]

Se reportan:

\[
kDN_{\mathrm{micro}}
=
\frac1N\sum_i kDN(x_i)
\]

y:

\[
kDN_{\mathrm{macro}}
=
\frac1C\sum_c
\left[
\frac1{n_c}
\sum_{i:y_i=c}kDN(x_i)
\right].
\]

También se reporta:

\[
kDN_{\mathrm{rara}}
\]

para la clase patológica.

kDN **no modifica las hipótesis principales**. Se usa para verificar si una limpieza local como Tomek reduce efectivamente la ambigüedad local.


In [ ]:
# ============================================================
# 8. kDN
# ============================================================
def kdn_metrics(X, y, k=5, rare_class=RARE_CLASS):
    n = len(y)
    k_eff = min(k, n - 1)

    nn = NearestNeighbors(
        n_neighbors=k_eff + 1,
        metric="euclidean"
    )
    nn.fit(X)

    _, indices = nn.kneighbors(X)

    # El primer vecino es la propia observación
    neigh_idx = indices[:, 1:]
    neigh_labels = y[neigh_idx]

    kdn = np.mean(
        neigh_labels != y[:, None],
        axis=1
    )

    classes = np.unique(y)

    class_means = {
        int(c): float(np.mean(kdn[y == c]))
        for c in classes
    }

    return {
        "kdn_micro": float(np.mean(kdn)),
        "kdn_macro": float(np.mean(list(class_means.values()))),
        "kdn_rare": class_means[int(rare_class)],
        **{f"kdn_class_{c}": v for c, v in class_means.items()}
    }


# 9. Tomek Links y controles

La condición de Tomek:

\[
TL(x_i,x_j)
\iff
y_i\ne y_j
\land NN(x_i)=x_j
\land NN(x_j)=x_i.
\]

Para la intervención principal:

- Tomek se aplica **sólo al entrenamiento**.
- Se usa `sampling_strategy="majority"`.
- La clase rara nunca se elimina por diseño de esta intervención.

Control `random_matched`:

Si Tomek elimina \(m\) observaciones de la mayoría, el control aleatorio elimina exactamente \(m\) observaciones mayoritarias.

Esto permite separar:

\[
\text{efecto de eliminar cantidad}
\]

de:

\[
\text{efecto de eliminar específicamente frontera}.
\]


In [ ]:
# ============================================================
# 9. Preprocesamiento
# ============================================================
def apply_tomek_majority(X, y):
    sampler = TomekLinks(sampling_strategy="majority")
    X_new, y_new = sampler.fit_resample(X, y)

    kept = np.asarray(sampler.sample_indices_)
    removed_mask = np.ones(len(y), dtype=bool)
    removed_mask[kept] = False
    removed_idx = np.where(removed_mask)[0]

    return X_new, y_new, removed_idx

def random_matched_undersample(X, y, n_remove, random_state):
    rng = np.random.default_rng(random_state)

    classes, counts = np.unique(y, return_counts=True)
    majority_class = classes[np.argmax(counts)]

    idx_majority = np.where(y == majority_class)[0]

    n_remove = min(int(n_remove), len(idx_majority) - 1)

    if n_remove <= 0:
        return X.copy(), y.copy(), np.array([], dtype=int)

    removed_idx = rng.choice(
        idx_majority,
        size=n_remove,
        replace=False
    )

    keep = np.ones(len(y), dtype=bool)
    keep[removed_idx] = False

    return X[keep], y[keep], removed_idx

def apply_smote_tomek(X, y, random_state):
    counts = pd.Series(y).value_counts()
    min_count = int(counts.min())

    k_smote = max(1, min(5, min_count - 1))

    smote = SMOTE(
        sampling_strategy="not majority",
        k_neighbors=k_smote,
        random_state=random_state
    )

    sampler = SMOTETomek(
        smote=smote,
        tomek=TomekLinks(sampling_strategy="all"),
        random_state=random_state
    )

    return sampler.fit_resample(X, y)


# 10. Clasificador D-min multiclase

Para cada clase:

\[
\mu_k=\frac1{n_k}\sum_{i:y_i=k}x_i.
\]

La regla D-min es:

\[
\hat y(x)=
\arg\min_k d(x,\mu_k).
\]

Bajo Gaussianas equiprobables y covarianza común isotrópica:

\[
\Sigma=\sigma^2I
\]

la regla MAP se reduce a D-min Euclidiano.

Para ROC-AUC de la clase rara se necesita un **score continuo**, no una etiqueta dura. Se define:

\[
s_r(x)=
\min_{j\ne r}d(x,\mu_j)
-
d(x,\mu_r).
\]

Así:

- \(s_r(x)>0\): evidencia relativa a favor de la clase rara.
- \(s_r(x)<0\): evidencia relativa a favor de alguna clase no rara.


In [ ]:
# ============================================================
# 10. D-min multiclase
# ============================================================
class DMinClassifier:
    def __init__(self, metric="euclidean", rare_class=RARE_CLASS):
        self.metric = metric
        self.rare_class = rare_class

    def fit(self, X, y):
        self.classes_, self.centroids_ = class_centroids(X, y)

        self.VI_ = None
        if self.metric == "mahalanobis":
            self.VI_ = inverse_covariance_ledoit(X)

        return self

    def _distances(self, X):
        return distance_matrix(
            X,
            self.centroids_,
            metric=self.metric,
            VI=self.VI_
        )

    def predict(self, X):
        D = self._distances(X)
        idx = np.argmin(D, axis=1)
        return self.classes_[idx]

    def rare_score(self, X):
        D = self._distances(X)

        rare_idx = np.where(
            self.classes_ == self.rare_class
        )[0][0]

        rare_d = D[:, rare_idx]

        other_cols = [
            j for j in range(D.shape[1])
            if j != rare_idx
        ]

        nearest_other_d = D[:, other_cols].min(axis=1)

        return nearest_other_d - rare_d


# 11. Métricas predictivas

Para la clase rara se usa one-vs-rest:

\[
y^{(r)}=
\begin{cases}
1,&y=r\\
0,&y\ne r
\end{cases}
\]

y se calculan:

\[
Se=\frac{TP}{TP+FN}
\]

\[
Sp=\frac{TN}{TN+FP}
\]

\[
PPV=\frac{TP}{TP+FP}
\]

\[
VPN=\frac{TN}{TN+FN}
\]

\[
BA=\frac{Se+Sp}{2}.
\]

Además:

- ROC-AUC
- PR-AUC
- Accuracy multiclase
- Balanced Accuracy multiclase

Las hipótesis confirmatorias usan el desempeño **rare-vs-rest**.


In [ ]:
# ============================================================
# 11. Métricas
# ============================================================
def evaluation_metrics(y_true, y_pred, rare_scores, rare_class=RARE_CLASS):
    y_true_bin = (y_true == rare_class).astype(int)
    y_pred_bin = (y_pred == rare_class).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true_bin,
        y_pred_bin,
        labels=[0, 1]
    ).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    npv = tn / (tn + fn) if (tn + fn) else np.nan
    ba_rare = 0.5 * (sensitivity + specificity)

    try:
        roc_auc = roc_auc_score(y_true_bin, rare_scores)
    except ValueError:
        roc_auc = np.nan

    try:
        pr_auc = average_precision_score(y_true_bin, rare_scores)
    except ValueError:
        pr_auc = np.nan

    return {
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "precision": float(precision),
        "npv": float(npv),
        "balanced_accuracy_rare": float(ba_rare),
        "roc_auc_rare": float(roc_auc),
        "pr_auc_rare": float(pr_auc),
        "accuracy_multiclass": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy_multiclass": float(
            balanced_accuracy_score(y_true, y_pred)
        )
    }


# 12. Validación repetida sin fuga de información

Para cada fold:

\[
Train
\rightarrow
\text{Mediana/MAD}_{Train}
\rightarrow
\text{preprocesamiento}_{Train}
\rightarrow
D\text{-min}
\]

mientras que:

\[
Test
\rightarrow
\text{transformación usando parámetros de Train}
\rightarrow
\text{evaluación}.
\]

El test nunca participa en:

- mediana;
- MAD;
- Tomek;
- SMOTE;
- selección aleatoria;
- centroides;
- covarianza.

### Unidad inferencial

Cada repetición produce predicciones **out-of-fold para las 2000 observaciones**.

Por ello, para cada repetición \(r\) se calcula una sola métrica:

\[
M_r.
\]

Las diferencias confirmatorias son:

\[
\Delta_r=
M_{\mathrm{Tomek},r}
-
M_{\mathrm{Baseline},r}.
\]

Así se obtienen \(R=10\) pares para la inferencia y no se tratan los folds como observaciones independientes.


In [ ]:
# ============================================================
# 12. Ejecución del experimento
# ============================================================
cv = RepeatedStratifiedKFold(
    n_splits=K,
    n_repeats=N_REPEATS,
    random_state=SEED
)

fold_audit_rows = []
geometry_fold_rows = []
kdn_fold_rows = []

# Acumuladores de predicciones OOF por repetición/método/métrica
oof_store = {}

split_id = 0

for train_idx, test_idx in cv.split(X, y):
    split_id += 1

    repeat_id = (split_id - 1) // K + 1
    fold_id = (split_id - 1) % K + 1

    Xtr_raw = X[train_idx]
    Xte_raw = X[test_idx]

    ytr = y[train_idx]
    yte = y[test_idx]

    # --------------------------------------------------------
    # Escalado robusto SOLO con train
    # --------------------------------------------------------
    scaler = MedianMADScaler()
    Xtr = scaler.fit_transform(Xtr_raw)
    Xte = scaler.transform(Xte_raw)

    # --------------------------------------------------------
    # Tomek primario
    # --------------------------------------------------------
    X_tomek, y_tomek, tomek_removed_idx = apply_tomek_majority(
        Xtr, ytr
    )

    # --------------------------------------------------------
    # Control aleatorio con igual número de eliminaciones
    # --------------------------------------------------------
    X_random, y_random, random_removed_idx = random_matched_undersample(
        Xtr,
        ytr,
        n_remove=len(tomek_removed_idx),
        random_state=SEED + split_id
    )

    # --------------------------------------------------------
    # Benchmark híbrido
    # --------------------------------------------------------
    X_smote, y_smote = apply_smote_tomek(
        Xtr,
        ytr,
        random_state=SEED + split_id
    )

    method_data = {
        "baseline": (Xtr, ytr),
        "tomek": (X_tomek, y_tomek),
        "random_matched": (X_random, y_random),
        "smote_tomek": (X_smote, y_smote)
    }

    # Auditoría del fold
    fold_audit_rows.append({
        "split_id": split_id,
        "repeat": repeat_id,
        "fold": fold_id,
        "n_train": len(ytr),
        "n_test": len(yte),
        "rare_train": int(np.sum(ytr == RARE_CLASS)),
        "rare_test": int(np.sum(yte == RARE_CLASS)),
        "class0_train": int(np.sum(ytr == 0)),
        "class1_train": int(np.sum(ytr == 1)),
        "class2_train": int(np.sum(ytr == 2)),
        "tomek_removed_total": int(len(tomek_removed_idx)),
        "tomek_removed_rare": int(
            np.sum(ytr[tomek_removed_idx] == RARE_CLASS)
        ) if len(tomek_removed_idx) else 0,
        "random_removed_total": int(len(random_removed_idx))
    })

    # --------------------------------------------------------
    # Cada método
    # --------------------------------------------------------
    for method, (Xm, ym) in method_data.items():

        # kDN es Euclidiano local y se calcula una vez por método/fold
        kdn = kdn_metrics(
            Xm, ym,
            k=K_KDN,
            rare_class=RARE_CLASS
        )

        kdn_fold_rows.append({
            "split_id": split_id,
            "repeat": repeat_id,
            "fold": fold_id,
            "method": method,
            **kdn
        })

        # Covarianza Ledoit-Wolf del conjunto procesado
        VI = inverse_covariance_ledoit(Xm)

        for metric in METRICS:
            this_VI = VI if metric == "mahalanobis" else None

            geo = geometry_metrics(
                Xm,
                ym,
                rare_class=RARE_CLASS,
                metric=metric,
                VI=this_VI
            )

            geometry_fold_rows.append({
                "split_id": split_id,
                "repeat": repeat_id,
                "fold": fold_id,
                "method": method,
                "metric": metric,
                "n_train_after": len(ym),
                "class0_after": int(np.sum(ym == 0)),
                "class1_after": int(np.sum(ym == 1)),
                "class2_after": int(np.sum(ym == 2)),
                **geo
            })

            clf = DMinClassifier(
                metric=metric,
                rare_class=RARE_CLASS
            ).fit(Xm, ym)

            pred = clf.predict(Xte)
            score = clf.rare_score(Xte)

            key = (repeat_id, method, metric)

            if key not in oof_store:
                oof_store[key] = {
                    "y_true": [],
                    "y_pred": [],
                    "score": []
                }

            oof_store[key]["y_true"].append(yte)
            oof_store[key]["y_pred"].append(pred)
            oof_store[key]["score"].append(score)

print("Experimento terminado.")


In [ ]:
# ============================================================
# 13. DataFrames base
# ============================================================
audit_df = pd.DataFrame(fold_audit_rows)
geo_fold_df = pd.DataFrame(geometry_fold_rows)
kdn_fold_df = pd.DataFrame(kdn_fold_rows)

display(audit_df.head())
print("Mínimo de casos raros por test fold:", audit_df["rare_test"].min())
print("Máximo de casos raros por test fold:", audit_df["rare_test"].max())
print("Máximo de raros eliminados por Tomek:", audit_df["tomek_removed_rare"].max())

audit_df.to_csv(RESULTS_DIR / "01_auditoria_folds.csv", index=False)
geo_fold_df.to_csv(RESULTS_DIR / "02_geometria_por_fold.csv", index=False)
kdn_fold_df.to_csv(RESULTS_DIR / "03_kdn_por_fold.csv", index=False)


# 14. Métricas predictivas por repetición

No se promedian sensibilidades de folds. Para cada repetición se concatenan todas las predicciones out-of-fold:

\[
\{(\hat y_i,s_i):i=1,\ldots,N\}.
\]

Con ellas se recalculan la matriz de confusión, sensibilidad, especificidad, BA, ROC-AUC y PR-AUC para toda la repetición.


In [ ]:
# ============================================================
# 14. Predicción OOF por repetición
# ============================================================
classification_repeat_rows = []

for (repeat_id, method, metric), store in oof_store.items():
    y_true = np.concatenate(store["y_true"])
    y_pred = np.concatenate(store["y_pred"])
    scores = np.concatenate(store["score"])

    mets = evaluation_metrics(
        y_true,
        y_pred,
        scores,
        rare_class=RARE_CLASS
    )

    classification_repeat_rows.append({
        "repeat": repeat_id,
        "method": method,
        "metric": metric,
        **mets
    })

class_repeat_df = pd.DataFrame(classification_repeat_rows)

display(class_repeat_df.head())

class_repeat_df.to_csv(
    RESULTS_DIR / "04_clasificacion_oof_por_repeticion.csv",
    index=False
)


In [ ]:
# ============================================================
# 15. Geometría y kDN por repetición
# ============================================================
geo_repeat_df = (
    geo_fold_df
    .groupby(["repeat", "method", "metric"], as_index=False)[
        [
            "D_intra_bal",
            "D_intra_rare",
            "D_inter_macro",
            "D_inter_rare",
            "G_macro",
            "G_rare",
            "D_intra_class_0",
            "D_intra_class_1",
            "D_intra_class_2"
        ]
    ]
    .mean()
)

kdn_repeat_df = (
    kdn_fold_df
    .groupby(["repeat", "method"], as_index=False)[
        [
            "kdn_micro",
            "kdn_macro",
            "kdn_rare",
            "kdn_class_0",
            "kdn_class_1",
            "kdn_class_2"
        ]
    ]
    .mean()
)

geo_repeat_df.to_csv(
    RESULTS_DIR / "05_geometria_por_repeticion.csv",
    index=False
)

kdn_repeat_df.to_csv(
    RESULTS_DIR / "06_kdn_por_repeticion.csv",
    index=False
)

display(geo_repeat_df.head())
display(kdn_repeat_df.head())


# 16. Hipótesis confirmatorias

Las hipótesis se conservan.

### Geométrica

\[
H_{0G}:
\operatorname{Mediana}
(G_{\mathrm{rara,post}}-G_{\mathrm{rara,pre}})
\le0
\]

\[
H_{1G}:
\operatorname{Mediana}
(G_{\mathrm{rara,post}}-G_{\mathrm{rara,pre}})
>0
\]

### Sensibilidad

\[
H_{0S}:
\operatorname{Mediana}
(Se_{post}-Se_{pre})
\le0
\]

### Exactitud balanceada

\[
H_{0BA}:
\operatorname{Mediana}
(BA_{post}-BA_{pre})
\le0
\]

### ROC-AUC

\[
H_{0AUC}:
\operatorname{Mediana}
(AUC_{post}-AUC_{pre})
\le0.
\]

Las cuatro comparaciones se ajustan con Holm.


In [ ]:
# ============================================================
# 16. Funciones de pares
# ============================================================
def paired_geo(variable, metric=PRIMARY_DISTANCE):
    pre = geo_repeat_df[
        (geo_repeat_df["method"] == "baseline") &
        (geo_repeat_df["metric"] == metric)
    ][["repeat", variable]].rename(columns={variable: "pre"})

    post = geo_repeat_df[
        (geo_repeat_df["method"] == "tomek") &
        (geo_repeat_df["metric"] == metric)
    ][["repeat", variable]].rename(columns={variable: "post"})

    out = pre.merge(post, on="repeat")
    out["delta"] = out["post"] - out["pre"]
    return out

def paired_class(variable, metric=PRIMARY_DISTANCE):
    pre = class_repeat_df[
        (class_repeat_df["method"] == "baseline") &
        (class_repeat_df["metric"] == metric)
    ][["repeat", variable]].rename(columns={variable: "pre"})

    post = class_repeat_df[
        (class_repeat_df["method"] == "tomek") &
        (class_repeat_df["metric"] == metric)
    ][["repeat", variable]].rename(columns={variable: "post"})

    out = pre.merge(post, on="repeat")
    out["delta"] = out["post"] - out["pre"]
    return out

def paired_kdn(variable):
    pre = kdn_repeat_df[
        kdn_repeat_df["method"] == "baseline"
    ][["repeat", variable]].rename(columns={variable: "pre"})

    post = kdn_repeat_df[
        kdn_repeat_df["method"] == "tomek"
    ][["repeat", variable]].rename(columns={variable: "post"})

    out = pre.merge(post, on="repeat")
    out["delta"] = out["post"] - out["pre"]
    return out

pairs = {
    "G_rare": paired_geo("G_rare"),
    "sensitivity": paired_class("sensitivity"),
    "balanced_accuracy_rare": paired_class("balanced_accuracy_rare"),
    "roc_auc_rare": paired_class("roc_auc_rare")
}


In [ ]:
# ============================================================
# 17. Wilcoxon unilateral + Holm
# ============================================================
def safe_wilcoxon_greater(post, pre):
    delta = np.asarray(post) - np.asarray(pre)

    if np.allclose(delta, 0):
        return np.nan, 1.0

    try:
        stat, p = wilcoxon(
            post,
            pre,
            alternative="greater",
            zero_method="wilcox"
        )
        return stat, p
    except ValueError:
        return np.nan, 1.0

hyp_rows = []

for variable, pair_df in pairs.items():
    stat, p = safe_wilcoxon_greater(
        pair_df["post"],
        pair_df["pre"]
    )

    hyp_rows.append({
        "variable": variable,
        "n_pairs": len(pair_df),
        "mean_pre": pair_df["pre"].mean(),
        "mean_post": pair_df["post"].mean(),
        "mean_delta": pair_df["delta"].mean(),
        "median_delta": pair_df["delta"].median(),
        "wilcoxon_stat": stat,
        "p_raw": p
    })

hyp_df = pd.DataFrame(hyp_rows)

reject, p_holm, _, _ = multipletests(
    hyp_df["p_raw"],
    alpha=0.05,
    method="holm"
)

hyp_df["p_holm"] = p_holm
hyp_df["reject_H0_alpha_0.05"] = reject

display(hyp_df)

hyp_df.to_csv(
    RESULTS_DIR / "07_hipotesis_wilcoxon_holm.csv",
    index=False
)


# 18. Bootstrap de las diferencias por repetición

Se utiliza como **intervalo de estabilidad interna**, no como sustituto de validación externa.

Para cada variable:

\[
\Delta_r=M_{post,r}-M_{pre,r}.
\]

Se remuestrean con reemplazo las \(R=10\) diferencias y se calcula:

\[
IC_{95\%}
=
[Q_{0.025}(\Delta^*),Q_{0.975}(\Delta^*)].
\]

Debido a que las repeticiones reutilizan el mismo dataset, este intervalo no debe interpretarse como un IC clínico poblacional.


In [ ]:
# ============================================================
# 18. Bootstrap
# ============================================================
def bootstrap_delta_ci(deltas, n_boot=20000, seed=SEED):
    deltas = np.asarray(deltas, dtype=float)
    rng = np.random.default_rng(seed)

    boot = np.empty(n_boot)

    for b in range(n_boot):
        sample = rng.choice(
            deltas,
            size=len(deltas),
            replace=True
        )
        boot[b] = np.mean(sample)

    return (
        float(np.mean(deltas)),
        float(np.quantile(boot, 0.025)),
        float(np.quantile(boot, 0.975))
    )

bootstrap_rows = []

for variable, pair_df in pairs.items():
    mean_delta, lo, hi = bootstrap_delta_ci(
        pair_df["delta"].values
    )

    bootstrap_rows.append({
        "variable": variable,
        "mean_delta": mean_delta,
        "IC95_low": lo,
        "IC95_high": hi,
        "IC95_above_zero": bool(lo > 0)
    })

bootstrap_df = pd.DataFrame(bootstrap_rows)

final_hyp_df = hyp_df.merge(
    bootstrap_df,
    on=["variable", "mean_delta"],
    how="left"
)

final_hyp_df["supported_by_both_criteria"] = (
    (final_hyp_df["p_holm"] < 0.05) &
    (final_hyp_df["IC95_low"] > 0)
)

display(final_hyp_df)

final_hyp_df.to_csv(
    RESULTS_DIR / "08_hipotesis_finales.csv",
    index=False
)


# 19. kDN como evidencia local

Para Tomek se espera, si la frontera local se vuelve menos ambigua:

\[
\Delta kDN_{\mathrm{macro}}<0,
\qquad
\Delta kDN_{\mathrm{rara}}<0.
\]

Esto se reporta de forma complementaria y no añade una nueva hipótesis confirmatoria.


In [ ]:
# ============================================================
# 19. Cambios de kDN
# ============================================================
kdn_pairs = {}

for variable in ["kdn_micro", "kdn_macro", "kdn_rare"]:
    p = paired_kdn(variable)
    kdn_pairs[variable] = p

    print("\n", variable)
    print("Pre :", p["pre"].mean())
    print("Post:", p["post"].mean())
    print("Delta:", p["delta"].mean())

kdn_change_df = pd.DataFrame([
    {
        "variable": name,
        "mean_pre": p["pre"].mean(),
        "mean_post": p["post"].mean(),
        "mean_delta": p["delta"].mean()
    }
    for name, p in kdn_pairs.items()
])

display(kdn_change_df)

kdn_change_df.to_csv(
    RESULTS_DIR / "09_cambios_kdn.csv",
    index=False
)


# 20. Comparación de métodos

Se comparan:

1. Baseline.
2. Tomek.
3. Random undersampling emparejado.
4. SMOTE+Tomek.

El control aleatorio es especialmente importante: si Tomek elimina el mismo número de observaciones que `random_matched` pero produce un resultado mejor, la diferencia puede asociarse a **dónde se eliminan los puntos**, no sólo a cuántos se eliminan.


In [ ]:
# ============================================================
# 20. Resumen por método con métrica primaria
# ============================================================
primary_results = class_repeat_df[
    class_repeat_df["metric"] == PRIMARY_DISTANCE
]

method_summary = (
    primary_results
    .groupby("method")[
        [
            "sensitivity",
            "specificity",
            "precision",
            "npv",
            "balanced_accuracy_rare",
            "roc_auc_rare",
            "pr_auc_rare",
            "accuracy_multiclass",
            "balanced_accuracy_multiclass"
        ]
    ]
    .agg(["mean", "std", "median"])
    .round(6)
)

display(method_summary)

method_summary.to_csv(
    RESULTS_DIR / "10_resumen_metodos_euclidiana.csv"
)


In [ ]:
# ============================================================
# 21. Auditoría de las seis métricas
# ============================================================
metric_summary = (
    class_repeat_df
    .groupby(["method", "metric"])[
        [
            "sensitivity",
            "specificity",
            "balanced_accuracy_rare",
            "roc_auc_rare",
            "pr_auc_rare",
            "balanced_accuracy_multiclass"
        ]
    ]
    .agg(["mean", "std"])
    .round(6)
)

display(metric_summary)

metric_summary.to_csv(
    RESULTS_DIR / "11_resumen_por_metrica.csv"
)


# 22. Visualizaciones sin PCA

Las gráficas 2D utilizan las dos primeras características originales, porque `shuffle=False` deja las primeras características dentro del bloque informativo.

Estas figuras son **descriptivas**. La conclusión sobre mejora de frontera no depende de que una proyección 2D “se vea más limpia”.


In [ ]:
# ============================================================
# 22. Figura: distribución de clases
# ============================================================
fig = plt.figure(figsize=(7, 5))

counts.plot(kind="bar")
plt.xlabel("Clase")
plt.ylabel("Número de observaciones")
plt.title("Distribución multiclase del dataset sintético")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()

fig.savefig(
    RESULTS_DIR / "fig_01_distribucion_clases.png",
    dpi=220,
    bbox_inches="tight"
)
plt.show()


In [ ]:
# ============================================================
# 23. Figura 2D de dos características originales
# ============================================================
scaler_vis = MedianMADScaler()
X_vis = scaler_vis.fit_transform(X)

fig = plt.figure(figsize=(8, 6))

markers = {0: "o", 1: "s", 2: "^"}

for c in np.unique(y):
    plt.scatter(
        X_vis[y == c, 0],
        X_vis[y == c, 1],
        s=18 if c != RARE_CLASS else 55,
        alpha=0.25 if c == 0 else 0.85,
        marker=markers[c],
        label=f"Clase {c}"
    )

plt.xlabel("Característica original 1 — escala Mediana/MAD")
plt.ylabel("Característica original 2 — escala Mediana/MAD")
plt.title("Vista 2D del espacio original — sin PCA")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()

fig.savefig(
    RESULTS_DIR / "fig_02_vista_2D_sin_PCA.png",
    dpi=220,
    bbox_inches="tight"
)
plt.show()


In [ ]:
# ============================================================
# 24. Figura pre/post Tomek en el primer split
# ============================================================
# Se reconstruye el primer split únicamente para visualización.

cv_vis = RepeatedStratifiedKFold(
    n_splits=K,
    n_repeats=1,
    random_state=SEED
)

train_idx, test_idx = next(cv_vis.split(X, y))

Xtr_raw = X[train_idx]
ytr = y[train_idx]

scaler = MedianMADScaler()
Xtr = scaler.fit_transform(Xtr_raw)

X_t, y_t, removed_idx = apply_tomek_majority(Xtr, ytr)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for c in np.unique(ytr):
    axes[0].scatter(
        Xtr[ytr == c, 0],
        Xtr[ytr == c, 1],
        s=18 if c != RARE_CLASS else 55,
        alpha=0.25 if c == 0 else 0.85,
        marker=markers[c],
        label=f"Clase {c}"
    )

if len(removed_idx):
    axes[0].scatter(
        Xtr[removed_idx, 0],
        Xtr[removed_idx, 1],
        s=70,
        facecolors="none",
        edgecolors="black",
        linewidths=1.2,
        label="Eliminados por Tomek"
    )

axes[0].set_title("Antes de Tomek")
axes[0].set_xlabel("Característica 1")
axes[0].set_ylabel("Característica 2")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.2)

for c in np.unique(y_t):
    axes[1].scatter(
        X_t[y_t == c, 0],
        X_t[y_t == c, 1],
        s=18 if c != RARE_CLASS else 55,
        alpha=0.25 if c == 0 else 0.85,
        marker=markers[c],
        label=f"Clase {c}"
    )

axes[1].set_title("Después de Tomek")
axes[1].set_xlabel("Característica 1")
axes[1].set_ylabel("Característica 2")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.2)

plt.tight_layout()

fig.savefig(
    RESULTS_DIR / "fig_03_pre_post_tomek.png",
    dpi=220,
    bbox_inches="tight"
)
plt.show()


In [ ]:
# ============================================================
# 25. Figura: deltas por repetición
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

for ax, (name, pair_df) in zip(
    axes.ravel(),
    pairs.items()
):
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.plot(
        pair_df["repeat"],
        pair_df["delta"],
        marker="o"
    )
    ax.set_title(f"Delta {name}")
    ax.set_xlabel("Repetición")
    ax.set_ylabel("Post - Pre")
    ax.grid(alpha=0.2)

plt.tight_layout()

fig.savefig(
    RESULTS_DIR / "fig_04_deltas_hipotesis.png",
    dpi=220,
    bbox_inches="tight"
)
plt.show()


In [ ]:
# ============================================================
# 26. Figura: kDN pre/post
# ============================================================
kdn_plot = kdn_repeat_df[
    kdn_repeat_df["method"].isin(["baseline", "tomek"])
]

fig = plt.figure(figsize=(8, 5))

for method, grp in kdn_plot.groupby("method"):
    plt.plot(
        grp["repeat"],
        grp["kdn_macro"],
        marker="o",
        label=method
    )

plt.xlabel("Repetición")
plt.ylabel("kDN macro")
plt.title("Ambigüedad local pre/post Tomek")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()

fig.savefig(
    RESULTS_DIR / "fig_05_kdn_macro.png",
    dpi=220,
    bbox_inches="tight"
)
plt.show()


# 27. Auditorías automáticas

Antes de interpretar resultados se verifican condiciones mínimas:

1. Cada fold de test contiene la clase rara.
2. Tomek no elimina casos raros.
3. Baseline y Tomek tienen el mismo número de repeticiones.
4. La inferencia usa 10 pares.
5. No hay valores NaN inesperados en las métricas confirmatorias.


In [ ]:
# ============================================================
# 27. Checks
# ============================================================
assert audit_df["rare_test"].min() >= 1, "Existe un fold sin clase rara."
assert audit_df["tomek_removed_rare"].max() == 0, "Tomek eliminó un caso raro."

for name, p in pairs.items():
    assert len(p) == N_REPEATS, f"{name}: número de pares incorrecto."
    assert not p[["pre", "post"]].isna().any().any(), f"{name}: contiene NaN."

print("AUDITORÍA SUPERADA")
print("Todos los folds contienen casos raros.")
print("Tomek no eliminó casos raros.")
print("Cada hipótesis tiene", N_REPEATS, "pares.")


In [ ]:
# ============================================================
# 28. Resumen automático para la tesina
# ============================================================
def fmt(x, d=6):
    return "NA" if pd.isna(x) else f"{x:.{d}f}"

lines = []

lines.append("RESUMEN AUTOMÁTICO DEL EXPERIMENTO")
lines.append("=" * 60)
lines.append(f"N = {N_SAMPLES}")
lines.append(f"d = {N_FEATURES}")
lines.append(f"K = {K}")
lines.append(f"R = {N_REPEATS}")
lines.append(f"Casos raros = {n_rare}")
lines.append(f"Prevalencia rara = {n_rare/N_SAMPLES:.4%}")
lines.append("")
lines.append("AUDITORÍA TOMЕК")
lines.append(f"Media eliminados por fold = {audit_df['tomek_removed_total'].mean():.3f}")
lines.append(f"Máximo de raros eliminados = {audit_df['tomek_removed_rare'].max()}")
lines.append("")
lines.append("HIPÓTESIS CONFIRMATORIAS")

for _, row in final_hyp_df.iterrows():
    lines.append(
        f"{row['variable']}: "
        f"pre={fmt(row['mean_pre'])}, "
        f"post={fmt(row['mean_post'])}, "
        f"delta={fmt(row['mean_delta'])}, "
        f"p_Holm={fmt(row['p_holm'])}, "
        f"IC95=[{fmt(row['IC95_low'])}, {fmt(row['IC95_high'])}], "
        f"respaldo={bool(row['supported_by_both_criteria'])}"
    )

lines.append("")
lines.append("kDN")
for _, row in kdn_change_df.iterrows():
    lines.append(
        f"{row['variable']}: "
        f"pre={fmt(row['mean_pre'])}, "
        f"post={fmt(row['mean_post'])}, "
        f"delta={fmt(row['mean_delta'])}"
    )

summary_text = "\n".join(lines)

print(summary_text)

with open(
    RESULTS_DIR / "12_resumen_automatico.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(summary_text)


In [ ]:
# ============================================================
# 29. Exportar pares confirmatorios
# ============================================================
for name, pair_df in pairs.items():
    pair_df.to_csv(
        RESULTS_DIR / f"pares_{name}.csv",
        index=False
    )


In [ ]:
# ============================================================
# 30. Crear ZIP y descargar
# ============================================================
zip_path = Path("/content/resultados_tesina_completos.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for path in RESULTS_DIR.rglob("*"):
        if path.is_file():
            z.write(path, arcname=path.name)

print("ZIP creado:", zip_path)

from google.colab import files
files.download(str(zip_path))


# Cómo interpretar los resultados

La conclusión debe seguir esta jerarquía:

### 1. Evidencia local
- número de observaciones eliminadas por Tomek;
- \(kDN_{\mathrm{micro}}\);
- \(kDN_{\mathrm{macro}}\);
- \(kDN_{\mathrm{rara}}\).

### 2. Evidencia geométrica global y específica
- \(D_{\mathrm{intra,bal}}\);
- \(D_{\mathrm{intra,rara}}\);
- \(D_{\mathrm{inter,macro}}\);
- \(D_{\mathrm{inter,rara}}\);
- \(G_{\mathrm{macro}}\);
- \(G_{\mathrm{rara}}\).

### 3. Evidencia predictiva
- sensibilidad;
- especificidad;
- precisión;
- VPN;
- exactitud balanceada;
- ROC-AUC;
- PR-AUC.

### 4. Evidencia estadística
- dirección de \(\Delta\);
- Wilcoxon;
- Holm;
- intervalo bootstrap de estabilidad.

## Regla de interpretación

No debe afirmarse que la frontera “mejoró” únicamente porque una figura 2D se vea más limpia.

Una afirmación fuerte requiere coherencia entre:

\[
\text{menor ambigüedad local}
\]

\[
\text{mejor geometría}
\]

y

\[
\text{mejor desempeño de D-min}.
\]

Si \(G_{\mathrm{rara}}\) mejora pero sensibilidad/BA/AUC no lo hacen, la conclusión correcta será:

> Tomek produjo una modificación geométrica detectable, pero dicha modificación no se tradujo en una mejora predictiva suficientemente consistente bajo D-min.

Esto sigue siendo un resultado científicamente válido.
